# 🚢 Notebook 1: The Bulkhead Pattern

Ships have **watertight bulkheads** — if one compartment floods, the others stay dry and the ship stays afloat.

In software: **isolate resources** (thread pools, connection pools) per downstream dependency so a failure in one does not drain resources from the others.

## 🛠️ Setup

```bash
cd 05-microservices/bulkhead
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Shared pool = one slow service sinks all

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def slow_service():
    time.sleep(2); return 'slow'
def fast_service():
    time.sleep(0.05); return 'fast'

shared = ThreadPoolExecutor(max_workers=4)

# Fill the shared pool with slow calls
slow_futures = [shared.submit(slow_service) for _ in range(4)]

# Now try 'fast' calls — they queue behind the slow ones!
t0 = time.time()
fast = shared.submit(fast_service).result()
print(f'fast call waited {time.time()-t0:.2f}s — the pool was full of slow work')


## Bulkheads: a separate pool per dependency

In [ ]:
pool_slow = ThreadPoolExecutor(max_workers=2, thread_name_prefix='slow')
pool_fast = ThreadPoolExecutor(max_workers=2, thread_name_prefix='fast')

# Saturate the slow-service pool
for _ in range(4):
    pool_slow.submit(slow_service)  # some will queue, that's fine

t0 = time.time()
print(pool_fast.submit(fast_service).result())
print(f'fast call waited {time.time()-t0:.2f}s — isolated pool stays healthy')


### Takeaways
- Think of each downstream as its **own compartment**.
- Give every compartment a **bounded** pool — bound = bulkhead thickness.
- Pair with circuit breaker + timeouts for full resilience.